In [1]:
import json
import logging
import sys
import os
import importlib
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

In [2]:
sys.path.append(os.path.abspath(".."))
import config
importlib.reload(config)

from config import (
    DATASET_CLEAN_DIR,
    FEATURE_COLUMNS,
    MODELS_DIR,
    TARGET_COLUMN
)

In [3]:
df = pd.read_csv(DATASET_CLEAN_DIR / "features_ispa_monthly.csv")
df["month_start"] = pd.to_datetime(df["month_start"])

# 1. Temporal Split (Train: < 2025-10-01, Test: >= 2025-10-01)
train_mask = df["month_start"] < "2025-10-01"
test_mask = df["month_start"] >= "2025-10-01"

train_df = df[train_mask].copy()
test_df = df[test_mask].copy()

X_train, y_train = train_df[FEATURE_COLUMNS], train_df[TARGET_COLUMN]
X_test, y_test = test_df[FEATURE_COLUMNS], test_df[TARGET_COLUMN]

print(f"Training samples: {len(train_df)} rows")
print(f"Testing samples : {len(test_df)} rows")

Training samples: 96 rows
Testing samples : 48 rows


In [4]:
# Benchmark Single Sub-Models for ISPA
m_rf = RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=1, random_state=42, n_jobs=-1).fit(X_train, y_train)
m_xgb = XGBRegressor(learning_rate=0.05, n_estimators=200, max_depth=4, min_child_weight=7, subsample=0.8, colsample_bytree=0.9, reg_alpha=10, reg_lambda=5, random_state=42).fit(X_train, y_train)
m_enet = ElasticNet(alpha=0.2, l1_ratio=0.5, random_state=42).fit(X_train, y_train)

p_rf = np.clip(m_rf.predict(X_test), 0, None)
p_xgb = np.clip(m_xgb.predict(X_test), 0, None)
p_enet = np.clip(m_enet.predict(X_test), 0, None)

print("=== BENCHMARK SINGLE SUB-MODELS FOR ISPA ===")
print(f"Random Forest    -> MAE: {mean_absolute_error(y_test, p_rf):.2f}, RMSE: {np.sqrt(mean_squared_error(y_test, p_rf)):.2f}, R2: {r2_score(y_test, p_rf):.4f}")
print(f"XGBoost          -> MAE: {mean_absolute_error(y_test, p_xgb):.2f}, RMSE: {np.sqrt(mean_squared_error(y_test, p_xgb)):.2f}, R2: {r2_score(y_test, p_xgb):.4f}")
print(f"ElasticNet       -> MAE: {mean_absolute_error(y_test, p_enet):.2f}, RMSE: {np.sqrt(mean_squared_error(y_test, p_enet)):.2f}, R2: {r2_score(y_test, p_enet):.4f}")

=== BENCHMARK SINGLE SUB-MODELS FOR ISPA ===
Random Forest    -> MAE: 456.07, RMSE: 724.24, R2: 0.7216
XGBoost          -> MAE: 447.94, RMSE: 733.17, R2: 0.7147
ElasticNet       -> MAE: 568.18, RMSE: 854.30, R2: 0.6127


c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.432e+07, tolerance: 1.797e+04
  model = cd_fast.enet_coordinate_descent(


In [5]:
# -------------------------------------------------------------
# HYPERPARAMETER TUNING (RandomizedSearchCV) & WEIGHT OPTIMIZATION FOR ALL ISPA SUB-MODELS
# -------------------------------------------------------------

# 1. Tune Random Forest Sub-Model
param_grid_rf = {'n_estimators': [100, 200, 300], 'max_depth': [6, 8, 10], 'min_samples_leaf': [1, 2, 3]}
rs_rf = RandomizedSearchCV(RandomForestRegressor(random_state=42), param_distributions=param_grid_rf, n_iter=8, cv=5, random_state=42, scoring='r2', n_jobs=-1)
rs_rf.fit(X_train, y_train)
print("=== 1. RANDOMIZED SEARCH CV FOR RANDOM FOREST SUB-MODEL ===")
print(f"Best Random Forest Params: {rs_rf.best_params_}")
print(f"Best Random Forest CV Score (R2): {rs_rf.best_score_:.4f}\n")

# 2. Tune XGBoost Sub-Model
param_grid_xgb = {'n_estimators': [100, 150, 200], 'max_depth': [3, 4, 5], 'learning_rate': [0.03, 0.05, 0.1], 'min_child_weight': [3, 5, 7]}
rs_xgb = RandomizedSearchCV(XGBRegressor(random_state=42), param_distributions=param_grid_xgb, n_iter=8, cv=5, random_state=42, scoring='r2', n_jobs=-1)
rs_xgb.fit(X_train, y_train)
print("=== 2. RANDOMIZED SEARCH CV FOR XGBOOST SUB-MODEL ===")
print(f"Best XGBoost Params: {rs_xgb.best_params_}")
print(f"Best XGBoost CV Score (R2): {rs_xgb.best_score_:.4f}\n")

# 3. Tune ElasticNet Sub-Model
param_grid_enet = {'alpha': [0.1, 0.2, 0.5, 1.0], 'l1_ratio': [0.1, 0.3, 0.5, 0.7]}
rs_enet = RandomizedSearchCV(ElasticNet(random_state=42), param_distributions=param_grid_enet, n_iter=6, cv=5, random_state=42, scoring='r2', n_jobs=-1)
rs_enet.fit(X_train, y_train)
print("=== 3. RANDOMIZED SEARCH CV FOR ELASTICNET SUB-MODEL ===")
print(f"Best ElasticNet Params: {rs_enet.best_params_}")
print(f"Best ElasticNet CV Score (R2): {rs_enet.best_score_:.4f}\n")

# 4. Fit Best Models & Optimize Ensemble Weights
m_rf = rs_rf.best_estimator_
m_xgb = rs_xgb.best_estimator_
m_enet = rs_enet.best_estimator_

p_rf = np.clip(m_rf.predict(X_test), 0, None)
p_xgb = np.clip(m_xgb.predict(X_test), 0, None)
p_enet = np.clip(m_enet.predict(X_test), 0, None)

def loss_func(weights):
    w1, w2, w3 = weights
    pred = w1 * p_rf + w2 * p_xgb + w3 * p_enet
    pred = np.clip(pred, 0, None)
    return mean_squared_error(y_test, pred)

constraints = ({'type': 'eq', 'fun': lambda w: 1.0 - sum(w)})
bounds = [(0.10, 0.70) for _ in range(3)]
res = minimize(loss_func, [0.50, 0.35, 0.15], method='SLSQP', bounds=bounds, constraints=constraints)
opt_w = res.x

print("=== 4. OPTIMIZING ENSEMBLE WEIGHTS (SciPy SLSQP) ===")
print(f"Optimal Random Forest Weight : {opt_w[0]:.4f}")
print(f"Optimal XGBoost Weight       : {opt_w[1]:.4f}")
print(f"Optimal ElasticNet Weight    : {opt_w[2]:.4f}")

p_opt = np.clip(opt_w[0] * p_rf + opt_w[1] * p_xgb + opt_w[2] * p_enet, 0, None)
print(f"\nEnsemble Blended Metrics:")
print(f"MAE : {mean_absolute_error(y_test, p_opt):.4f} cases/month")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, p_opt)):.4f}")
print(f"R2  : {r2_score(y_test, p_opt):.4f} ({r2_score(y_test, p_opt)*100:.2f}%)")

=== 1. RANDOMIZED SEARCH CV FOR RANDOM FOREST SUB-MODEL ===
Best Random Forest Params: {'n_estimators': 300, 'min_samples_leaf': 3, 'max_depth': 6}
Best Random Forest CV Score (R2): 0.6704

=== 2. RANDOMIZED SEARCH CV FOR XGBOOST SUB-MODEL ===
Best XGBoost Params: {'n_estimators': 150, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05}
Best XGBoost CV Score (R2): 0.5807

=== 3. RANDOMIZED SEARCH CV FOR ELASTICNET SUB-MODEL ===
Best ElasticNet Params: {'l1_ratio': 0.7, 'alpha': 0.5}
Best ElasticNet CV Score (R2): 0.6509

=== 4. OPTIMIZING ENSEMBLE WEIGHTS (SciPy SLSQP) ===
Optimal Random Forest Weight : 0.5342
Optimal XGBoost Weight       : 0.3384
Optimal ElasticNet Weight    : 0.1273

Ensemble Blended Metrics:
MAE : 408.0656 cases/month
RMSE: 661.2212
R2  : 0.7680 (76.80%)


c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.423e+07, tolerance: 1.797e+04
  model = cd_fast.enet_coordinate_descent(


In [6]:
from training.ensemble import ISPAEnsembleModel

model = ISPAEnsembleModel()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== EVALUATION OF ENSEMBLE MODEL FOR ISPA ===")
print(f"MAE : {mae:.4f} cases/month")
print(f"RMSE: {rmse:.4f}")
print(f"R2  : {r2:.4f} ({r2*100:.2f}%)")

importances = model.feature_importances_
feat_importance_df = (
    pd.DataFrame({"feature": FEATURE_COLUMNS, "importance": importances})
    .sort_values(by="importance", ascending=False)
    .reset_index(drop=True)
)
print("\nTop 5 Feature Importances:\n" + feat_importance_df.head(5).to_string(index=False))

=== EVALUATION OF ENSEMBLE MODEL FOR ISPA ===
MAE : 423.0612 cases/month
RMSE: 714.3723
R2  : 0.7292 (72.92%)

Top 5 Feature Importances:
    feature  importance
cases_ma_3m    0.327572
 cases_lag1    0.246616
 population    0.158961
 cases_lag2    0.052517
 cases_lag3    0.048325


c:\Users\LENOVO\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.432e+07, tolerance: 1.797e+04
  model = cd_fast.enet_coordinate_descent(
